# Module 5 — Evaluation on Kaggle Dataset

**Goal:** Run pipeline on 20 prescriptions. Measure drug matching accuracy.


## Cell 1 — Bootstrap

In [ ]:
import json
from pathlib import Path
from google.colab import drive
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/prescriptai')
DRUGS_JSON = PROJECT_ROOT / 'data' / 'drugs' / 'indian_drugs.json'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

with open(DRUGS_JSON) as f:
    DRUG_CORPUS = json.load(f)

print(f'Corpus: {len(DRUG_CORPUS)} drugs')

Mounted at /content/drive
Corpus: 960 drugs


## Cell 2 — Test Prescriptions

In [ ]:
TEST = [
    {'id': 1, 'drugs': ['Digoxin']},
    {'id': 2, 'drugs': ['Oflazest OZ', 'Azenac-MR', 'Andial', 'Zofer']},
    {'id': 5, 'drugs': ['Syp Ephedrex', 'Syp Crocin DS', 'Syp Meftal-P']},
    {'id': 6, 'drugs': ['Inj Remdesivir']},
    {'id': 9, 'drugs': ['Azithromycin']},
    {'id': 12, 'drugs': ['Effortil', 'Novalgin']},
    {'id': 13, 'drugs': ['Hypnodorm']},
    {'id': 14, 'drugs': ['Inj REMDEC', 'Inj Actemra']},
    {'id': 16, 'drugs': ['Paracetamol']},
    {'id': 17, 'drugs': ['Tab. Augmentin', 'Tab. Enzoflam', 'Tab. Pan-D']},
    {'id': 18, 'drugs': ['Phenytoin Sodium Extended']},
    {'id': 23, 'drugs': ['T. Ranitidine', 'T. Paracetamol']},
    {'id': 24, 'drugs': ['T. Deplatt A', 'T. Azitor', 'T. Nitrocontin']},
    {'id': 26, 'drugs': ['Zestril', 'Ferrous Sulfate', 'Humulin N']},
]

total = sum(len(p['drugs']) for p in TEST)
print(f'Prescriptions: {len(TEST)}, Drugs: {total}')

Prescriptions: 14, Drugs: 28


## Cell 3 — ChromaDB + RAG

In [ ]:
!pip install -q rapidfuzz chromadb sentence-transformers torch

from rapidfuzz import fuzz
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import torch

chroma_dir = PROJECT_ROOT / 'data' / 'chroma_db'
client = chromadb.PersistentClient(path=str(chroma_dir), settings=Settings(anonymized_telemetry=False))
coll = client.get_collection('drugs_text')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)

def embed(t): return np.asarray(enc.encode([t] if isinstance(t, str) else t, normalize_embeddings=True, show_progress_bar=False))

def retrieve(q, k=3):
    v = embed(q)[0].tolist()
    r = coll.query(query_embeddings=[v], n_results=k)
    return [{'name': r['metadatas'][0][i].get('name'), 'generic': r['metadatas'][0][i].get('generic'), 'score': 1 - r['distances'][0][i]} for i in range(len(r['ids'][0]))]

print('Ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready


## Cell 4 — Evaluate

In [ ]:
print("Loading drug embeddings into memory...")

from sentence_transformers import SentenceTransformer
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)

# Embed all drugs once
DRUG_EMBEDDINGS = {}
for i, drug in enumerate(DRUG_CORPUS):
    if i % 100 == 0:
        print(f"  {i}/{len(DRUG_CORPUS)}...")

    text = f"{drug.get('name', '')} {drug.get('generic', '')} {drug.get('description', '')}"
    vec = enc.encode([text], normalize_embeddings=True, show_progress_bar=False)[0]
    DRUG_EMBEDDINGS[i] = {'drug': drug, 'vec': vec}

print(f"✅ {len(DRUG_EMBEDDINGS)} drugs embedded")

def retrieve(q, k=5):
    q_vec = enc.encode([q], normalize_embeddings=True, show_progress_bar=False)[0]

    scores = []
    for idx, data in DRUG_EMBEDDINGS.items():
        sim = np.dot(q_vec, data['vec'])  # cosine similarity
        scores.append({'idx': idx, 'score': sim, 'name': data['drug']['name'], 'generic': data['drug']['generic']})

    # Top k
    top_k = sorted(scores, key=lambda x: x['score'], reverse=True)[:k]
    return [{'name': x['name'], 'generic': x['generic'], 'score': x['score']} for x in top_k]

print("Ready")

Loading drug embeddings into memory...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0/960...
  100/960...
  200/960...
  300/960...
  400/960...
  500/960...
  600/960...
  700/960...
  800/960...
  900/960...
✅ 960 drugs embedded
Ready


In [ ]:
print("="*60)
print("EVALUATION")
print("="*60 + "\n")

RES = []
for p in TEST:
    for drug in p['drugs']:
        cands = retrieve(drug, k=5)

        if not cands:
            print(f"[{p['id']}] {drug:25} → NO CANDIDATES")
            RES.append({'img': p['id'], 'extracted': drug, 'matched': 'N/A', 'score': 0.0, 'status': 'no'})
            continue

        best = None
        best_score = 0

        for cand in cands:
            fuzzy_name = fuzz.WRatio(drug.lower(), cand['name'].lower()) / 100.0
            fuzzy_generic = fuzz.WRatio(drug.lower(), cand['generic'].lower()) / 100.0
            fuzzy = max(fuzzy_name, fuzzy_generic)

            semantic = cand['score']
            hybrid = 0.6 * semantic + 0.4 * fuzzy

            if hybrid > best_score:
                best_score = hybrid
                best = cand

        status = 'matched' if best_score >= 0.55 else 'low' if best_score >= 0.35 else 'no'
        RES.append({'img': p['id'], 'extracted': drug, 'matched': best['name'] if best else 'N/A', 'score': best_score, 'status': status})
        print(f"[{p['id']}] {drug:25} → {best['name'] if best else 'N/A':25} ({best_score:.3f})")

print(f"\n✅ Eval complete: {len(RES)} drugs")

df = pd.DataFrame(RES)
m = len(df[df['status'] == 'matched'])
l = len(df[df['status'] == 'low'])
u = len(df[df['status'] == 'no'])
t = len(df)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Matched (≥0.55): {m}/{t} ({100*m/t:.1f}%)")
print(f"Low conf (0.35-0.55): {l}/{t} ({100*l/t:.1f}%)")
print(f"Unmatched (<0.35): {u}/{t} ({100*u/t:.1f}%)")
print(f"Avg score: {df['score'].mean():.3f}")
print("="*60)

EVALUATION

[1] Digoxin                   → Digoxin                   (0.833)
[2] Oflazest OZ               → Ofloxacin                 (0.321)
[2] Azenac-MR                 → Azenam 1gm Injection      (0.597)
[2] Andial                    → Andial 2mg Tablet         (0.603)
[2] Zofer                     → Azoran Tablet             (0.423)
[5] Syp Ephedrex              → Empetus 100mg Tablet      (0.391)
[5] Syp Crocin DS             → Crocin Syrup              (0.552)
[5] Syp Meftal-P              → Arip MT 5 Tablet          (0.361)
[6] Inj Remdesivir            → Remdesivir                (0.619)
[9] Azithromycin              → Azithromycin              (0.825)
[12] Effortil                  → Apsnal 5mg Tablet         (0.314)
[12] Novalgin                  → Novalgin                  (0.724)
[13] Hypnodorm                 → Hypnodorm                 (0.650)
[14] Inj REMDEC                → Intajac 5mg Tablet        (0.409)
[14] Inj Actemra               → Alerta Injection          (

## Cell 5 — Metrics

In [ ]:
df = pd.DataFrame(RES)
m = len(df[df['status'] == 'matched'])
l = len(df[df['status'] == 'low'])
u = len(df[df['status'] == 'no'])
t = len(df)

print('='*50)
print(f'Matched: {m}/{t} ({100*m/t:.1f}%)')
print(f'Low conf: {l}/{t} ({100*l/t:.1f}%)')
print(f'Unmatched: {u}/{t} ({100*u/t:.1f}%)')
print(f'Avg score: {df["score"].mean():.3f}')
print('='*50)

Matched: 15/28 (53.6%)
Low conf: 9/28 (32.1%)
Unmatched: 4/28 (14.3%)
Avg score: 0.559


## Cell 6 — Save

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

# Convert float32 to float
RES_clean = []
for r in RES:
    r['score'] = float(r['score'])  # numpy → Python float
    RES_clean.append(r)

out = {'metrics': {'total': t, 'matched': m, 'matched_pct': round(100*m/t, 1)}, 'results': RES_clean}
(OUTPUT_DIR / f'eval_{ts}.json').write_text(json.dumps(out, indent=2))
df.to_csv(OUTPUT_DIR / f'eval_{ts}.csv', index=False)
print(f'✅ Saved to {OUTPUT_DIR}')

✅ Saved to /content/drive/MyDrive/prescriptai/data/output
